In [ ]:
# RuModernBERT Domain Experiments

# This notebook evaluates a transformer-based CWE classifier using RuModernBERT.
# The same Russian, non-Russian, and combined training conditions used
# for the classical models are evaluated on the fixed Russian test set.

In [1]:
import gc
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    precision_recall_fscore_support,
)

from transformers import Trainer

from src.config import (
    TEXT_COL,
    LABEL_COL,
    ID_COL,
    CWE_LABELS,
    MODEL_ID,
    TRANSFORMER_RUNS_DIR,
    RESULTS_MAIN_DIR,
    RESULTS_PER_CLASS_DIR,
    RESULTS_PREDICTIONS_DIR,
)

from src.data import (
    load_preprocessed_data,
    assert_no_group_overlap,
)

from src.evaluation import (
    domain_generalisation_gap,
    combined_training_gain,
)

from src.transformer import (
    ID2LABEL,
    set_global_seed,
    print_device_info,
    make_tokenizer,
    make_hf_dataset,
    make_data_collator,
    calculate_class_weights,
    make_weighted_loss,
    make_model,
    compute_metrics,
    make_training_arguments,
    cleanup_cuda,
)

In [ ]:
## Initialise the Experiment and Construct Training Conditions

# Random seeds and hardware settings are initialised, the fixed partitions are loaded,
# and the Russian, non-Russian, and combined development datasets are constructed.
# Leakage isolation from the Russian test set is verified.

In [2]:
set_global_seed()
print_device_info()


ru_df, nonru_df = (
    load_preprocessed_data()
)


ru_train = (
    ru_df[
        ru_df["split"] == "train"
    ]
    .copy()
)

ru_val = (
    ru_df[
        ru_df["split"] == "val"
    ]
    .copy()
)

ru_test = (
    ru_df[
        ru_df["split"] == "test"
    ]
    .copy()
)


ru_dev = pd.concat(
    [ru_train, ru_val],
    ignore_index=True
)

nonru_dev = (
    nonru_df[
        nonru_df["split"]
        .isin(["train", "val"])
    ]
    .copy()
)

combined_dev = pd.concat(
    [
        ru_dev,
        nonru_dev
    ],
    ignore_index=True
)


assert_no_group_overlap(
    nonru_dev,
    ru_test
)

assert_no_group_overlap(
    combined_dev,
    ru_test
)

PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
GPU memory: 8.0 GB
BF16: True
FP16: False


True

In [ ]:
## Examine Tokenised Sequence Lengths

# The complete modelling corpus is tokenised without truncation
# to inspect the sequence-length distribution and quantify
# the proportion of descriptions exceeding the fixed maximum length of 256 tokens.

In [3]:
tokenizer = make_tokenizer()

all_text = pd.concat(
    [
        ru_df[TEXT_COL],
        nonru_df[TEXT_COL]
    ],
    ignore_index=True
)


lengths = []

for start in range(
    0,
    len(all_text),
    512
):

    encoded = tokenizer(
        all_text.iloc[
            start:start + 512
        ].tolist(),

        truncation=False,
        padding=False
    )

    lengths.extend(
        len(ids)
        for ids
        in encoded["input_ids"]
    )


lengths = np.array(lengths)


print(
    "Median:",
    np.median(lengths)
)

print(
    "95th percentile:",
    np.percentile(
        lengths,
        95
    )
)

print(
    ">256:",
    (lengths > 256).mean()
)

Median: 55.0
95th percentile: 85.0
>256: 8.910432334176854e-05


In [ ]:
## Pilot Training for Epoch Selection

# RuModernBERT is fine-tuned on the Russian training partition and evaluated
# after each epoch on the Russian validation partition.
# Class-weighted cross-entropy is used to account for the imbalanced CWE distribution.

In [4]:
train_ds = make_hf_dataset(
    ru_train,
    tokenizer
)

val_ds = make_hf_dataset(
    ru_val,
    tokenizer
)


pilot_model = make_model()

pilot_weights = (
    calculate_class_weights(
        ru_train
    )
)


pilot_args = (
    make_training_arguments(
        output_dir=(
            TRANSFORMER_RUNS_DIR
            / "pilot"
        ),
        num_epochs=4,
        evaluation=True
    )
)


pilot_trainer = Trainer(
    model=pilot_model,

    args=pilot_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    processing_class=tokenizer,

    data_collator=(
        make_data_collator(
            tokenizer
        )
    ),

    compute_metrics=compute_metrics,

    compute_loss_func=(
        make_weighted_loss(
            pilot_weights
        )
    )
)


pilot_trainer.train()

Map:   0%|          | 0/10114 [00:00<?, ? examples/s]

Map:   0%|          | 0/2166 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy
1,3.615597,0.957815,0.837171,0.862774,0.864266
2,0.971097,0.579421,0.851588,0.884305,0.882733
3,0.527410,0.666353,0.859539,0.890907,0.890120
4,0.226551,0.752768,0.860225,0.894514,0.894737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2532, training_loss=1.3351635081696474, metrics={'train_runtime': 611.2684, 'train_samples_per_second': 66.184, 'train_steps_per_second': 4.142, 'total_flos': 2173925864857320.0, 'train_loss': 1.3351635081696474, 'epoch': 4.0})

In [ ]:
## Select the Number of Training Epochs

# Validation history is examined and the epoch with the highest Macro-F1 is selected.
# This fixed number of epochs is subsequently used for all final transformer training conditions.

In [5]:
history = pd.DataFrame(
    pilot_trainer.state.log_history
)


validation_history = (
    history[
        history[
            "eval_macro_f1"
        ].notna()
    ]
    [
        [
            "epoch",
            "eval_macro_f1",
            "eval_weighted_f1",
            "eval_accuracy",
            "eval_loss",
        ]
    ]
)


BEST_EPOCHS = int(
    round(
        validation_history.loc[
            validation_history[
                "eval_macro_f1"
            ].idxmax(),
            "epoch"
        ]
    )
)


print(
    "Selected epochs:",
    BEST_EPOCHS
)

validation_history

Selected epochs: 4


,epoch,eval_macro_f1,eval_weighted_f1,eval_accuracy,eval_loss
1,1.0,0.837171,0.862774,0.864266,0.957815
3,2.0,0.851588,0.884305,0.882733,0.579421
5,3.0,0.859539,0.890907,0.890120,0.666353
7,4.0,0.860225,0.894514,0.894737,0.752768


In [ ]:
## Release Pilot-Run Resources

# Objects used only for validation-based epoch selection are removed
# and GPU memory is cleared before the final transformer experiments.

In [6]:
del pilot_trainer
del pilot_model
del train_ds
del val_ds

cleanup_cuda()

In [ ]:
## Define the Final Transformer Evaluation Procedure

# A common training and evaluation routine is defined for all domain conditions.
# Each run trains a fresh RuModernBERT model, evaluates it on the fixed Russian test set,
# and returns aggregate metrics, per-CWE metrics, and record-level predictions.

In [7]:
def run_final_transformer(
    condition_name,
    train_df
):

    set_global_seed()

    train_dataset = (
        make_hf_dataset(
            train_df,
            tokenizer
        )
    )

    test_dataset = (
        make_hf_dataset(
            ru_test,
            tokenizer
        )
    )

    model = make_model()

    class_weights = (
        calculate_class_weights(
            train_df
        )
    )

    args = (
        make_training_arguments(
            output_dir=(
                TRANSFORMER_RUNS_DIR
                /
                condition_name
                .lower()
                .replace(" ", "_")
            ),
            num_epochs=BEST_EPOCHS,
            evaluation=False
        )
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        data_collator=(
            make_data_collator(
                tokenizer
            )
        ),
        compute_loss_func=(
            make_weighted_loss(
                class_weights
            )
        )
    )


    start = time.time()

    trainer.train()

    runtime = (
        time.time()
        -
        start
    )


    output = trainer.predict(
        test_dataset
    )

    y_true = output.label_ids

    y_pred = np.argmax(
        output.predictions,
        axis=-1
    )


    metrics = compute_metrics(
        (
            output.predictions,
            y_true
        )
    )


    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(len(CWE_LABELS))
            ),
            zero_division=0
        )
    )


    per_class = pd.DataFrame({
        "CWE":
            CWE_LABELS,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support,
    })


    predictions = pd.DataFrame({
        ID_COL:
            ru_test[ID_COL].values,

        "true_cwe":
            [
                ID2LABEL[int(i)]
                for i in y_true
            ],

        "prediction":
            [
                ID2LABEL[int(i)]
                for i in y_pred
            ],

        "training_condition":
            condition_name,
    })


    result = {
        "model":
            "RuModernBERT-base",

        "training_condition":
            condition_name,

        "train_records":
            len(train_df),

        "test_records":
            len(ru_test),

        **metrics,

        "epochs":
            BEST_EPOCHS,

        "runtime_minutes":
            runtime / 60,
    }


    del trainer
    del model
    del train_dataset
    del test_dataset

    cleanup_cuda()


    return (
        result,
        per_class,
        predictions
    )

In [ ]:
## Russian-to-Russian Transformer Evaluation

# RuModernBERT is trained on the full Russian development set and evaluated
# on the fixed Russian test set, providing the in-domain transformer result.

In [8]:
ru_result, ru_per_class, ru_predictions = (
    run_final_transformer(
        "RU",
        ru_dev
    )
)

ru_result

Map:   0%|          | 0/12280 [00:00<?, ? examples/s]

Map:   0%|          | 0/2171 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss
768,3.350368
1536,0.927025
2304,0.527875
3072,0.234413


{'model': 'RuModernBERT-base',
 'training_condition': 'RU',
 'train_records': 12280,
 'test_records': 2171,
 'macro_f1': 0.8974600021163472,
 'weighted_f1': 0.9100632563576463,
 'accuracy': 0.9101796407185628,
 'epochs': 4,
 'runtime_minutes': 11.513085254033406}

In [ ]:
## Non-Russian-to-Russian Transformer Evaluation

# RuModernBERT is trained exclusively on non-Russian development data
# and evaluated on the same Russian test set to measure cross-domain transfer.

In [9]:
nonru_result, nonru_per_class, nonru_predictions = (
    run_final_transformer(
        "Non-RU",
        nonru_dev
    )
)

nonru_result

Map:   0%|          | 0/41628 [00:00<?, ? examples/s]

Map:   0%|          | 0/2171 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss
2602,2.277826
5204,0.727848
7806,0.419611
10408,0.194952


{'model': 'RuModernBERT-base',
 'training_condition': 'Non-RU',
 'train_records': 41628,
 'test_records': 2171,
 'macro_f1': 0.8588478686720405,
 'weighted_f1': 0.8810744097773466,
 'accuracy': 0.8820819898664211,
 'epochs': 4,
 'runtime_minutes': 38.17685568332672}

In [ ]:
## Combined-to-Russian Transformer Evaluation

# RuModernBERT is trained on the combined Russian and non-Russian
# development data and evaluated on the fixed Russian test set.

In [10]:
combined_result, combined_per_class, combined_predictions = (
    run_final_transformer(
        "Combined",
        combined_dev
    )
)

combined_result

Map:   0%|          | 0/53908 [00:00<?, ? examples/s]

Map:   0%|          | 0/2171 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss
3370,2.048214
6740,0.730079
10110,0.419178
13480,0.192254


{'model': 'RuModernBERT-base',
 'training_condition': 'Combined',
 'train_records': 53908,
 'test_records': 2171,
 'macro_f1': 0.902413801565121,
 'weighted_f1': 0.9165386388928151,
 'accuracy': 0.9166282818977429,
 'epochs': 4,
 'runtime_minutes': 72.83614036639531}

In [ ]:
## Compare Transformer Training Conditions

# The aggregate results from the three transformer
# training conditions are combined into a single comparison table.

In [11]:
transformer_results = pd.DataFrame([
    ru_result,
    nonru_result,
    combined_result
])

transformer_results

,model,training_condition,train_records,test_records,macro_f1,weighted_f1,accuracy,epochs,runtime_minutes
0,RuModernBERT-base,RU,12280,2171,0.897460,0.910063,0.910180,4,11.513085
1,RuModernBERT-base,Non-RU,41628,2171,0.858848,0.881074,0.882082,4,38.176856
2,RuModernBERT-base,Combined,53908,2171,0.902414,0.916539,0.916628,4,72.836140


In [ ]:
## Quantify Transformer Domain Effects

# The transformer results are summarised using the domain generalisation gap
# and combined-training gain, directly matching the central comparisons used for the classical models.

In [13]:
values = (
    transformer_results
    .set_index(
        "training_condition"
    )["macro_f1"]
)


transformer_comparison = pd.DataFrame([{

    "RU_to_RU_macro_f1":
        values["RU"],

    "NonRU_to_RU_macro_f1":
        values["Non-RU"],

    "Combined_to_RU_macro_f1":
        values["Combined"],

    "domain_generalisation_gap":
        domain_generalisation_gap(
            values["RU"],
            values["Non-RU"]
        ),

    "combined_training_gain":
        combined_training_gain(
            values["RU"],
            values["Combined"]
        )
}])


transformer_comparison

,RU_to_RU_macro_f1,NonRU_to_RU_macro_f1,Combined_to_RU_macro_f1,domain_generalisation_gap,combined_training_gain
0,0.89746,0.858848,0.902414,0.038612,0.004954


In [ ]:
## Save Aggregate Transformer Results

# Overall RuModernBERT performance and the central cross-domain
# comparison measures are exported for the main results analysis.

In [14]:
transformer_results.to_csv(
    RESULTS_MAIN_DIR
    / "RuModernBERT_domain_results.csv",
    index=False
)

transformer_comparison.to_csv(
    RESULTS_MAIN_DIR
    / "RuModernBERT_domain_comparison.csv",
    index=False
)

In [ ]:
## Save Transformer Predictions and Per-Class Metrics

# Record-level predictions and per-CWE evaluation metrics for all three training conditions
# are saved for subsequent class-level and error analysis.

In [15]:
ru_predictions.to_csv(
    RESULTS_PREDICTIONS_DIR
    / "RuModernBERT_predictions_RU.csv",
    index=False
)

ru_per_class.to_csv(
    RESULTS_PER_CLASS_DIR
    / "RuModernBERT_per_class_RU.csv",
    index=False
)

nonru_predictions.to_csv(
    RESULTS_PREDICTIONS_DIR
    / "RuModernBERT_predictions_NonRU.csv",
    index=False
)

nonru_per_class.to_csv(
    RESULTS_PER_CLASS_DIR
    / "RuModernBERT_per_class_NonRU.csv",
    index=False
)

combined_predictions.to_csv(
    RESULTS_PREDICTIONS_DIR
    / "RuModernBERT_predictions_Combined.csv",
    index=False
)

combined_per_class.to_csv(
    RESULTS_PER_CLASS_DIR
    / "RuModernBERT_per_class_Combined.csv",
    index=False
)

print("RuModernBERT results saved.")

RuModernBERT results saved.
